# Jour 2 : Bag of Words, TFiDF et calcul de proximité de vecteur

In [9]:
!pip install pandas numpy scikit-learn spacy


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [10]:
!python -m spacy download fr_core_news_sm

/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:991: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 9.8 MB/s  0:00:01m eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation suc

In [11]:
import string

import numpy as np
import pandas as pd
import os
import fr_core_news_sm
from scipy.sparse.linalg import svds
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import manhattan_distances, euclidean_distances, cosine_similarity, linear_kernel


/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")
/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:991: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 11040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


In [13]:
# récupération du jeu de données
try:
    data = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_anthologia_graeca_fr.csv")
except FileNotFoundError:
    !wget -P corpus https://raw.githubusercontent.com/alexiaschn/dhsi-2026/main/notebooks/corpus/epigrammes_anthologia_graeca_fr.csv
    data = pd.read_csv(f"{os.getcwd()}/corpus/epigrammes_anthologia_graeca_fr.csv")


data.head()



,epigramme_id,book,fragment,urn,author_fr,text_fr
0,7.114,7,114,urn:cts:greekLit:tlg7000.tlg001.ag:7.114,Diogène Laërce,"Tu voulais, Héraclide, laisser aux hommes le b..."
1,7.115,7,115,urn:cts:greekLit:tlg7000.tlg001.ag:7.115,Diogène Laërce,"« Tu as vécu en chien, Antisthène. - C'est que..."
2,7.116,7,116,urn:cts:greekLit:tlg7000.tlg001.ag:7.116,Diogène Laërce,"« Diogène, allons, dis-moi quel destin t'a rav..."
3,7.120,7,120,urn:cts:greekLit:tlg7000.tlg001.ag:7.120,Xénophane,On dit qu'un jour il passait devant un jeune c...
4,7.171,7,171,urn:cts:greekLit:tlg7000.tlg001.ag:7.171,Mnasalcès,Ici aussi l'oiseau sacré reposera son aile rap...


Pour cet exercice on ne se sert plus que du texte 

In [14]:
corpus = data.text_fr
corpus

0     Tu voulais, Héraclide, laisser aux hommes le b...
1     « Tu as vécu en chien, Antisthène. - C'est que...
2     « Diogène, allons, dis-moi quel destin t'a rav...
3     On dit qu'un jour il passait devant un jeune c...
4     Ici aussi l'oiseau sacré reposera son aile rap...
5     Moi, Alcimènés, jadis j'écartais l'etourneau e...
6     D'elles-mêmes, le soir, les vaches sont revenu...
7     Non, jamais plus, mélodieuse sauterelle, Hélio...
8     A la sauterelle, rossignol des guérets, à la c...
9     Moi qui jadis, en renvoyant le sons, répondais...
10    Jamais plus de tes ailes bruissantes tu ne cha...
11    Sauterelle, toi qui trompes mes regrets et apa...
12    Tu ne jettes plus, posée sur les feuilles vert...
13    Tu ne vas plus t'élancer à travers les profond...
14    Les flots et les vagues houleuses m'ont jeté s...
Name: text_fr, dtype: object

In [ ]:
# prétraitement = tokenisation spaCy

spacy_pipeline = fr_core_news_sm.load(disable=["parser", "ner", "lemmatizer"])

def split_into_tokens_spacy(text) :
    doc = spacy_pipeline(text)
    return [w.text for w in doc]

def split_into_tokens_wosw_spacy(text):
    doc = spacy_pipeline(text)
    return [t.text for t in doc if not t.is_stop]

## 2. 👜 Représentation "sac de mots" pondérée par la fréquence

La transformation en vecteurs (sac de mots) est réalisée à l'aide de la bibliothèque Scikit-learn et de `CountVectorizer`. `CountVectorizer` a de nombreuses options, qui sont détaillées ici : https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

`CountVectorizer` a un paramètre nommé `tokenizer` qui permet de passer une fonction qui découpe le texte en tokens et effectue éventuellement d'autres prétraitements. Si aucune fonction de tokénisation n’est fournie, par défaut les tokens sont composés de séquences de deux caractères alphanumériques ou plus, la ponctuation est ignorée et considérée comme séparateur de token.

Le paramètre `min_df` permet de spécifier la proportion minimale des documents dans lesquels un token doit se trouver. Ainsi, `min_df=0.01` signifie que les tokens qui se trouvent dans moins de 1% des documents (ici, les épigrammes) ne seront pas pris en compte.

In [16]:
bow_transformer1  = CountVectorizer(tokenizer=split_into_tokens_spacy,
                                    lowercase = True,
                                    min_df=0.01)
# Fit --> "apprentissage" du vocabulaire
bow_transformer1.fit(corpus)

/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


CountVectorizer(min_df=0.01,
                tokenizer=<function split_into_tokens_spacy at 0x7b2fc78bdee0>)

In [17]:
# Vocabulaire
voc1 = bow_transformer1.get_feature_names_out()
print(voc1)

['!' "'" ',' '-' '-moi' '.' ':' ';' '?' 'a' 'abondante' 'accents' 'afin'
 'agiles' 'agreste' 'agréable' 'ai' 'aile' 'ailes' 'aimé' 'air' 'ait'
 'alcimènés' 'alkis' 'allait' 'allons' 'amer' 'ami' 'amour' 'antisthène'
 'apaises' 'appartenait' 'apte' 'arracher' 'arrêtant' 'arrête' 'as'
 'attrait' 'au' 'aussi' 'aussitôt' 'autour' 'autrefois' 'aux' 'avaient'
 'avais' 'avec' 'avoir' 'barques' 'besoin' 'bien' 'bistonie' 'blondes'
 'bornes' 'bouillonnantes' 'bras' 'bruissantes' 'bruit' 'bête' 'bûcherons'
 "c'" 'cailloux' 'calculs' 'car' 'ce' 'celle' 'ces' 'ceux' 'chair'
 'changé' 'chanson' 'chant' 'chantais' 'chanter' 'chanteras' 'chants'
 'charmé' 'chasse' 'cheville' 'chez' 'chien' 'chêne' 'ciel' 'cigale'
 'clyménos' 'comme' 'comment' 'commune' 'compassion' 'confiance'
 'convaicu' 'couvertes' 'cri' 'crier' 'cris' 'crocs' 'cœur' "d'" 'dans'
 'danser' 'dauphin' 'de' 'demeure' 'des' 'destin' 'devant' 'diogène'
 'dipsas' 'dira' 'dis' 'dit' 'diviseras' 'donnerai' 'dort' 'dos' 'double'
 'doute' 'du

In [18]:
corpus_bow = bow_transformer1.transform(corpus)
corpus_bow

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 701 stored elements and shape (15, 415)>

In [20]:
corpus_bow_df = pd.DataFrame(corpus_bow.toarray(), columns=voc1)
corpus_bow_df

,!,',",",-,-moi,.,:,;,?,a,...,élevé,épargné,étable,étais,étendu,éternel,étrange,été,être,ô
0,0,0,8,0,0,2,1,1,0,0,...,0,0,0,0,0,0,0,1,1,0
1,1,0,2,3,0,4,0,0,0,0,...,0,0,0,2,0,0,0,0,0,0
2,0,0,3,1,1,1,0,0,1,2,...,0,0,0,0,0,0,0,0,0,0
3,0,0,3,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,1,3,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,8,0,0,3,0,0,0,1,...,1,0,0,0,0,0,0,0,0,0
6,0,0,6,1,0,1,1,0,0,1,...,0,0,1,0,0,1,0,0,0,0
7,0,0,3,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,6,0,0,1,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
9,0,0,10,0,0,2,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


Quel constat peut-on faire de cette représentation ? 
Quels sont les tokens les plus représentés ? 

Comment est-ce qu'on pourrait améliorer cette vectorisation ? (comment pourrait-on obtenir des vecteurs plus denses ?)

In [ ]:
# écrire la fonction de lemmatisation avec spaCy sur le modèle de la fonction spaCy de tokenisation

def split_into_lemmas_spacy(text):
    # here
    pass

In [ ]:
# ajouter l'antidictionnaire français de NLTK à la pipeline CountVectorizer avec l'option stopwords
import nltk
nltk.download('stopwords')
nltk.corpus.stopwords.words('french')


bow_transformer2  = CountVectorizer(tokenizer=split_into_lemmas_spacy,
                                    lowercase = True,
                                    min_df=0.01,
                                    # ici
                                    )
# Fit --> "apprentissage" du vocabulaire
bow_transformer2.fit(corpus)

[nltk_data] Downloading package stopwords to /home/schn/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


['au',
 'aux',
 'avec',
 'ce',
 'ces',
 'dans',
 'de',
 'des',
 'du',
 'elle',
 'en',
 'et',
 'eux',
 'il',
 'ils',
 'je',
 'la',
 'le',
 'les',
 'leur',
 'lui',
 'ma',
 'mais',
 'me',
 'même',
 'mes',
 'moi',
 'mon',
 'ne',
 'nos',
 'notre',
 'nous',
 'on',
 'ou',
 'par',
 'pas',
 'pour',
 'qu',
 'que',
 'qui',
 'sa',
 'se',
 'ses',
 'son',
 'sur',
 'ta',
 'te',
 'tes',
 'toi',
 'ton',
 'tu',
 'un',
 'une',
 'vos',
 'votre',
 'vous',
 'c',
 'd',
 'j',
 'l',
 'à',
 'm',
 'n',
 's',
 't',
 'y',
 'été',
 'étée',
 'étées',
 'étés',
 'étant',
 'étante',
 'étants',
 'étantes',
 'suis',
 'es',
 'est',
 'sommes',
 'êtes',
 'sont',
 'serai',
 'seras',
 'sera',
 'serons',
 'serez',
 'seront',
 'serais',
 'serait',
 'serions',
 'seriez',
 'seraient',
 'étais',
 'était',
 'étions',
 'étiez',
 'étaient',
 'fus',
 'fut',
 'fûmes',
 'fûtes',
 'furent',
 'sois',
 'soit',
 'soyons',
 'soyez',
 'soient',
 'fusse',
 'fusses',
 'fût',
 'fussions',
 'fussiez',
 'fussent',
 'ayant',
 'ayante',
 'ayantes',


In [ ]:
# compléter la série (extraction du vocabulaire, transformation du corpus en BoW, afficher le dataFrame du nouveau BoW)

## Pondération TF-iDF




In [24]:
tfidf_vectorizer = TfidfVectorizer(tokenizer=split_into_tokens_spacy)
tfidf_vectorizer.fit(corpus)

/home/schn/Documents/GitLab/phd/.venv/lib/python3.12/site-packages/sklearn/feature_extraction/text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


TfidfVectorizer(tokenizer=<function split_into_tokens_spacy at 0x7b2fc78bdee0>)

In [28]:
corpus_tfidf = tfidf_vectorizer.transform(corpus)
tfidf_feat_names = tfidf_vectorizer.get_feature_names_out()

corpus_tfid_df = pd.DataFrame(corpus_tfidf.toarray(), columns=tfidf_feat_names).round(2)
corpus_tfid_df

,!,',",",-,-moi,.,:,;,?,a,...,élevé,épargné,étable,étais,étendu,éternel,étrange,été,être,ô
0,0.00,0.00,0.34,0.00,0.00,0.09,0.09,0.08,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.13,0.13,0.00
1,0.13,0.00,0.08,0.30,0.00,0.17,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.26,0.00,0.00,0.00,0.00,0.00,0.00
2,0.00,0.00,0.21,0.17,0.19,0.07,0.00,0.00,0.19,0.24,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,0.00,0.00,0.15,0.00,0.00,0.05,0.11,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,0.00,0.18,0.17,0.00,0.00,0.11,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,0.00,0.00,0.28,0.00,0.00,0.11,0.00,0.00,0.00,0.06,...,0.11,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
6,0.00,0.00,0.32,0.13,0.00,0.05,0.12,0.00,0.00,0.09,...,0.00,0.00,0.17,0.00,0.00,0.17,0.00,0.00,0.00,0.00
7,0.00,0.00,0.18,0.00,0.00,0.12,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
8,0.00,0.00,0.33,0.00,0.00,0.06,0.12,0.11,0.00,0.09,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
9,0.00,0.00,0.40,0.00,0.00,0.08,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [ ]:
# refaire la pipeline avec la lemmatisation et les stopwords de nltk dans la pipeline TfidfVectorizer

# Calculs de distance (/proximité)



In [29]:
cos_sim = cosine_similarity(corpus_tfidf)
print(cos_sim)

[[1.         0.18212743 0.09328803 0.09888427 0.12877372 0.20929594
  0.20010314 0.16832386 0.2004976  0.25106631 0.16429388 0.22866336
  0.19727076 0.18291959 0.17566897]
 [0.18212743 1.         0.20023387 0.23189026 0.13499657 0.12262235
  0.11627551 0.12394001 0.14436509 0.09840273 0.0894291  0.10318637
  0.13606922 0.10679236 0.12859806]
 [0.09328803 0.20023387 1.         0.11663194 0.0531584  0.12840416
  0.17792359 0.06365365 0.16024071 0.1142412  0.06409812 0.19790131
  0.08212686 0.1390908  0.2590686 ]
 [0.09888427 0.23189026 0.11663194 1.         0.16533387 0.13881599
  0.13500927 0.08762074 0.14010548 0.1240653  0.08811618 0.08766038
  0.1891098  0.11606494 0.12831408]
 [0.12877372 0.13499657 0.0531584  0.16533387 1.         0.18023645
  0.12143764 0.14790877 0.12617979 0.11385275 0.10458168 0.11758552
  0.17263363 0.17023105 0.14829731]
 [0.20929594 0.12262235 0.12840416 0.13881599 0.18023645 1.
  0.23567378 0.21865917 0.26150386 0.24892584 0.14508824 0.30234065
  0.1421853 

In [45]:
def get_most_similar_epigram(epigram_index):
    print(f"L'épigramme le plus proche de '{corpus[epigram_index]}'")
    # extraire la ligne contenant les scores de similarité pour cet épigramme : retourne tuple (position, score)
    sim_scores = list(enumerate(cos_sim[epigram_index]))
    # on organise la liste pour que le score le plus élevé (valeur en position 1 du tuple) soit en tête de liste
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # comme le score le plus élevé est avec lui-même, on ne prend la liste qu'à partir du 2e élément
    sim_scores = sim_scores[1:]
    return corpus[sim_scores[0][0]]

print(get_most_similar_epigram(1))

L'épigramme le plus proche de '« Tu as vécu en chien, Antisthène. - C'est que j'étais né pour déchirer le cœur avec des mots et non avec des crocs. - Mais tu es mort en chien que tu étais né, dira sans doute quelqu'un. - Eh quoi ! il fait bien qu'il y en ait pour servir de guide chez Hadès. »'
On dit qu'un jour il passait devant un jeune chien qu'on frappait, il en eut compassion et prononça ces paroles : « Arrête et ne frappe plus, car sûrement c'est l'âme d'un ami, que j'ai reconnue en l'entendant crier. »


In [ ]:
# Réécrire la fonction get_most_similar_epigram() pour qu'elle retourne les 3 épigrammes les plus proches avec leur score

Il existe d'autres façon de calculer la proximité entre deux vecteurs, par exemple:

- Distance L1 ou Manhattan 
- Distance L2 ou euclidienne



In [46]:
# reproduire le calcul des distances avec l'une ou l'autre distance à l'aide des fonctions pré-importées de scikit-learn :  manhattan_distances() ou euclidean_distances()

l1_distances = manhattan_distances(corpus_tfidf)
print(l1_distances)

[[ 0.         10.58499673 10.04300624 10.98411054 10.66291335 11.65377446
  10.23285136 10.35181515 10.13816463 10.12520071  9.95544651 11.35928714
  10.18764654 11.88308674 11.37053893]
 [10.58499673  0.          9.22144227  9.84687489 10.88038333 12.75783489
  11.21114567 10.99019616 10.78990568 11.36425637 10.88325701 12.8306298
  10.59856381 12.88932923 11.82438735]
 [10.04300624  9.22144227  0.          9.38818335 10.14488158 11.30569102
   8.95135516 10.04294947  9.36118479  9.88203761  9.46245204 10.37527211
   9.5611364  10.86256849  9.81023405]
 [10.98411054  9.84687489  9.38818335  0.          9.90509862 12.19804624
  10.55129809 10.70860219 10.3074987  11.00472999 10.51025721 12.57294198
   9.68630461 12.08249073 11.7589903 ]
 [10.66291335 10.88038333 10.14488158  9.90509862  0.         11.82985277
  10.81260676  9.88918211 10.62134249 10.95693507 10.22383669 12.34233902
   9.64767134 11.69752316 11.50840485]
 [11.65377446 12.75783489 11.30569102 12.19804624 11.82985277  0.


In [47]:
def get_most_similar_epigram2(epigram_index):
    print(f"L'épigramme le plus proche de '{corpus[epigram_index]}'")
    # extraire la ligne contenant les scores de similarité pour cet épigramme : retourne tuple (position, score)
    sim_scores = list(enumerate(l1_distances[epigram_index]))
    # on organise la liste pour que le score le plus élevé (valeur en position 1 du tuple) soit en tête de liste
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    # comme le score le plus élevé est avec lui-même, on ne prend la liste qu'à partir du 2e élément
    sim_scores = sim_scores[1:]
    return corpus[sim_scores[0][0]]

print(get_most_similar_epigram2(1))

L'épigramme le plus proche de '« Tu as vécu en chien, Antisthène. - C'est que j'étais né pour déchirer le cœur avec des mots et non avec des crocs. - Mais tu es mort en chien que tu étais né, dira sans doute quelqu'un. - Eh quoi ! il fait bien qu'il y en ait pour servir de guide chez Hadès. »'
Sauterelle, toi qui trompes mes regrets et apaises mon sommeil, sauterelle, muse agreste au vol mélodieux, imitation naturelle de la lyre, dis-moi quelque chant aimé en frappant de tes pattes tes ailes sonores, afin de m'arracher au souci de mes peines qui m'enlèvent le sommeil, cigale, toi qui sais les chants qui trompent l'amour. A ton réveil je te donnerai comme récompense du poireau toujours vert et des plantes humides de rosée, que tu diviseras de tes lèvres.
